# Euploid DINO Classifier — Colab Pipeline

End-to-end runner for the temporal attention classifier on frozen DINO ViT-S/16 features.

**Before running**, make sure you have in your Google Drive:
1. The DINO student checkpoint (`.pth`).
2. The `dino_training/` folder (with `models.py`, `dataset.py`, etc.).
3. The videos folder — one subfolder per embryo, named `<id>_..._E` or `_A`, one video inside each.

## 1. Mount Drive & clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone -b claude/embryo-classification-dino-MEbiv https://github.com/erkankalafat/euploid-dino-classifier.git || (cd euploid-dino-classifier && git pull)
%cd /content/euploid-dino-classifier
!pip install -q -r requirements.txt

## 2. Configure paths (EDIT THESE)

In [ ]:
import os

# --- EDIT to match your Drive layout ---
DRIVE_ROOT          = '/content/drive/MyDrive'
VIDEOS_DRIVE        = f'{DRIVE_ROOT}/embryo_videos'              # folder of per-embryo subfolders
DINO_CKPT           = f'{DRIVE_ROOT}/dino_training/checkpoint.pth'
DINO_TRAINING_ROOT  = f'{DRIVE_ROOT}'                             # parent of dino_training/
CHECKPOINT_DIR      = f'{DRIVE_ROOT}/euploid_ckpts'               # resumable training ckpts
FEATURES_DIR        = f'{DRIVE_ROOT}/euploid_features'            # cached DINO features

# Local working copies (faster I/O than Drive)
LOCAL_VIDEOS        = '/content/videos'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(LOCAL_VIDEOS, exist_ok=True)
print('paths ok')

## 3. Copy videos from Drive to local disk
Drive I/O is slow; copying once speeds everything downstream.

In [ ]:
!rsync -a --info=progress2 "{VIDEOS_DRIVE}/" "{LOCAL_VIDEOS}/"
!ls "{LOCAL_VIDEOS}" | head

## 4. Build manifest

In [ ]:
!python -m data.build_manifest --videos-root "{LOCAL_VIDEOS}" --out data/manifest.csv
import pandas as pd; pd.read_csv('data/manifest.csv').head()

## 5. Preview sampled frames (sanity check)

In [ ]:
!python -m scripts.preview_frames --n-embryos 3 --num-frames 40 --stride 10
!ls logs/preview

In [ ]:
# Display a grid from the first previewed embryo
import glob, matplotlib.pyplot as plt
from PIL import Image
dirs = sorted(glob.glob('logs/preview/*'))
if dirs:
    imgs = sorted(glob.glob(dirs[0] + '/*.png'))
    fig, axes = plt.subplots(5, 8, figsize=(16, 10))
    for ax, p in zip(axes.flat, imgs):
        ax.imshow(Image.open(p)); ax.set_title(p.split('/')[-1], fontsize=7); ax.axis('off')
    plt.tight_layout(); plt.show()

## 6. Extract DINO features (resumable)
Writes to `FEATURES_DIR` on Drive; skips already-cached embryos on rerun.

In [ ]:
!python -m feature_extraction.extract_dino_features \
    --manifest data/manifest.csv \
    --features-dir "{FEATURES_DIR}" \
    --dino-ckpt "{DINO_CKPT}" \
    --dino-training-root "{DINO_TRAINING_ROOT}" \
    --num-frames 40 --stride 10

### Sanity: intra- vs inter-embryo cosine similarity

In [ ]:
import torch, glob, random
import torch.nn.functional as F
files = sorted(glob.glob(f'{FEATURES_DIR}/*.pt'))[:5]
embs = [torch.load(f)['features'] for f in files]
def cos(a,b): return F.cosine_similarity(a,b,dim=-1).mean().item()
intra = [cos(e[0:1], e[1:2]) for e in embs]
inter = [cos(embs[i][0:1], embs[j][0:1]) for i in range(len(embs)) for j in range(len(embs)) if i!=j]
print('intra mean:', sum(intra)/len(intra))
print('inter mean:', sum(inter)/len(inter))

## 7. Point config at Drive features dir

In [ ]:
import yaml, pathlib
cfg_path = pathlib.Path('configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg['data']['features_dir'] = FEATURES_DIR
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)

## 8. Overfit sanity test (32 embryos)
Head should reach ~100% train AUC — confirms wiring.

In [ ]:
import pandas as pd
df = pd.read_csv('data/manifest.csv')
df.groupby('label').head(16).to_csv('data/manifest_overfit.csv', index=False)
# quick config override
import yaml, pathlib
cfg = yaml.safe_load(open('configs/default.yaml'))
cfg['data']['manifest'] = 'data/manifest_overfit.csv'
cfg['train']['epochs'] = 30
cfg['train']['folds'] = 2
pathlib.Path('configs/overfit.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
!python train.py --config configs/overfit.yaml --checkpoint-dir /content/overfit_ckpts

## 9. Full 5-fold CV training (resumable on Drive)
If the Colab runtime disconnects, just re-run this cell — it resumes from the last checkpoint per fold.

In [ ]:
!python train.py --config configs/default.yaml --checkpoint-dir "{CHECKPOINT_DIR}"

## 10. Evaluate a fold + attention plots

In [ ]:
!python eval.py --config configs/default.yaml \
    --checkpoint "{CHECKPOINT_DIR}/fold0.pt" \
    --plot-dir logs/attention
!ls logs/attention | head

In [ ]:
import json, glob
for f in sorted(glob.glob(f'{CHECKPOINT_DIR}/fold*_best.json')):
    print(f, json.load(open(f)))
summary = f'{CHECKPOINT_DIR}/cv_summary.json'
import os
if os.path.exists(summary):
    print('CV SUMMARY:', json.load(open(summary)))